In [4]:
import pandas as pd

F_DATA = "DS_BPE_2024_data.csv"
F_META = "DS_BPE_2024_metadata.csv"
bpe = pd.read_csv(F_DATA, sep=";", low_memory=False)
meta = pd.read_csv(F_META, sep=";", low_memory=False)

print("bpe shape:", bpe.shape)
print("meta shape:", meta.shape)
bpe.head(3)
print(bpe.columns.tolist())
print(meta.columns.tolist())

# Un coup d'œil aux domaines d'équipements dans les données
bpe["FACILITY_DOM"].value_counts().head(10)




bpe shape: (2300480, 8)
meta shape: (42077, 4)
['GEO', 'GEO_OBJECT', 'FACILITY_DOM', 'FACILITY_SDOM', 'FACILITY_TYPE', 'BPE_MEASURE', 'TIME_PERIOD', 'OBS_VALUE']
['COD_VAR', 'LIB_VAR', 'COD_MOD', 'LIB_MOD']


FACILITY_DOM
A     657757
D     505190
F     411123
B     375200
C     178231
G      72710
E      58460
_T     41809
Name: count, dtype: int64

In [5]:
# Repérer les sous-domaines liés à la santé
meta_sante = meta[meta["LIB_VAR"].str.contains("Santé", case=False, na=False)]
meta_sante[["COD_VAR", "LIB_VAR", "COD_MOD", "LIB_MOD"]].head(20)


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD


In [6]:
# Chercher dans la colonne LIB_MOD (libellé des sous-catégories)
meta_sante = meta[meta["LIB_MOD"].str.contains("sant", case=False, na=False)]
meta_sante[["COD_VAR", "LIB_VAR", "COD_MOD", "LIB_MOD"]].head(20)


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD
4,FACILITY_DOM,1er niveau de la classification des types d'éq...,D,Santé et action sociale
19,FACILITY_SDOM,2e niveau de la classification des types d'équ...,D1,Etablissements et services de santé
112,FACILITY_TYPE,3e niveau de la classification des types d'équ...,D108,Centre de santé
147,FACILITY_TYPE,3e niveau de la classification des types d'équ...,D113,Maison de santé pluridisciplinaire
163,FACILITY_TYPE,3e niveau de la classification des types d'équ...,D115,Services de santé maternelle et infantile
165,FACILITY_TYPE,3e niveau de la classification des types d'équ...,C603,Centre dispensant de la formation continue agr...
213,FACILITY_TYPE,3e niveau de la classification des types d'équ...,F109,Parcours sportif/santé
1453,GEO,Géographie,247300817,Communauté de communes Les Versants d'Aime
4532,GEO,Géographie,80114,Rosières-en-Santerre
5276,GEO,Géographie,80680,Rosières-en-Santerre


In [7]:
# Étape 4 - Filtrer les équipements santé
codes_sante = ["D1", "D108", "D113", "D115"]
bpe_sante = bpe[(bpe["FACILITY_DOM"] == "D") & (bpe["FACILITY_SDOM"].isin(codes_sante))].copy()

print("Nombre de lignes Santé :", len(bpe_sante))
bpe_sante.head()


Nombre de lignes Santé : 58638


,GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE
20,04,REG,D,D1,D104,FACILITIES,2024,35
23,01,REG,D,D1,D103,FACILITIES,2024,4
112,24,REG,D,D1,D107,FACILITIES,2024,18
283,94,REG,D,D1,D108,FACILITIES,2024,9
291,27,REG,D,D1,D108,FACILITIES,2024,149


In [8]:
# Étape 5 - Agréger le nombre d'équipements santé par commune

# On regroupe par code géographique GEO (chaque GEO = une commune)
bpe_agg = (
    bpe_sante.groupby("GEO")["BPE_MEASURE"]
    .sum()
    .reset_index()
    .rename(columns={"GEO": "code_commune", "BPE_MEASURE": "total_equipements_sante"})
)

print("Nombre de communes trouvées :", len(bpe_agg))
bpe_agg.head(10)


Nombre de communes trouvées : 8355


,code_commune,total_equipements_sante
0,001,FACILITIESFACILITIESFACILITIESFACILITIESFACILI...
1,00152,FACILITIESFACILITIES
2,00154,FACILITIESFACILITIESFACILITIES
3,00159,FACILITIESFACILITIES
4,00160,FACILITIESFACILITIES
5,00161,FACILITIESFACILITIES
6,00162,FACILITIESFACILITIESFACILITIESFACILITIES
7,00163,FACILITIESFACILITIES
8,00164,FACILITIESFACILITIESFACILITIES
9,00165,FACILITIESFACILITIES


In [9]:
# Vérifier le type de la colonne
print("Type BPE_MEASURE :", bpe_sante["BPE_MEASURE"].dtype)
print(bpe_sante["BPE_MEASURE"].unique()[:10])  # aperçu des premières valeurs

# Si les valeurs sont textuelles comme 'FACILITIES', on remplace par 1 (chaque ligne = un équipement)
bpe_sante["BPE_MEASURE"] = 1

# Regrouper à nouveau par commune
bpe_agg = (
    bpe_sante.groupby("GEO")["BPE_MEASURE"]
    .sum()
    .reset_index()
    .rename(columns={"GEO": "code_commune", "BPE_MEASURE": "total_equipements_sante"})
)

print("Nombre de communes trouvées :", len(bpe_agg))
bpe_agg.head(10)


Type BPE_MEASURE : object
['FACILITIES']
Nombre de communes trouvées : 8355


,code_commune,total_equipements_sante
0,001,16
1,00152,2
2,00154,3
3,00159,2
4,00160,2
5,00161,2
6,00162,4
7,00163,2
8,00164,3
9,00165,2


In [12]:
# Étape 7 - Normaliser le score de santé (entre 0 et 1)
bpe_agg["score_sante"] = (
    bpe_agg["total_equipements_sante"] - bpe_agg["total_equipements_sante"].min()
) / (
    bpe_agg["total_equipements_sante"].max()
    - bpe_agg["total_equipements_sante"].min()
)

print("✅ Score de santé calculé !")
display(bpe_agg.head(10))


# Étape 8 - Catégoriser le niveau de santé (gestion automatique des doublons)
try:
    bpe_agg["niveau_sante"] = pd.qcut(
        bpe_agg["total_equipements_sante"],
        q=4,
        labels=["faible", "moyen", "bon", "excellent"],
        duplicates='drop'
    )
except ValueError:
    # Si encore un souci, on réduit à 3 classes
    bpe_agg["niveau_sante"] = pd.qcut(
        bpe_agg["total_equipements_sante"],
        q=3,
        labels=["faible", "moyen", "bon"],
        duplicates='drop'
    )

print("✅ Niveaux de santé créés (avec tolérance sur doublons) !")
display(bpe_agg.head(10))



# Étape 9 - Sauvegarder le score santé
bpe_agg[["code_commune", "score_sante", "niveau_sante"]].to_csv("sante_score.csv", index=False)
print("💾 Fichier enregistré : sante_score.csv")


✅ Score de santé calculé !


,code_commune,total_equipements_sante,score_sante
0,001,16,0.400000
1,00152,2,0.000000
2,00154,3,0.028571
3,00159,2,0.000000
4,00160,2,0.000000
5,00161,2,0.000000
6,00162,4,0.057143
7,00163,2,0.000000
8,00164,3,0.028571
9,00165,2,0.000000


✅ Niveaux de santé créés (avec tolérance sur doublons) !


,code_commune,total_equipements_sante,score_sante,niveau_sante
0,001,16,0.400000,bon
1,00152,2,0.000000,faible
2,00154,3,0.028571,faible
3,00159,2,0.000000,faible
4,00160,2,0.000000,faible
5,00161,2,0.000000,faible
6,00162,4,0.057143,moyen
7,00163,2,0.000000,faible
8,00164,3,0.028571,faible
9,00165,2,0.000000,faible


💾 Fichier enregistré : sante_score.csv
